In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
baslangic_zamani = time.time()

spark = SparkSession.builder \
    .appName("PySpark_Baseline") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


ulkeler_ve_dosyalar = {
    "Türkiye": "Datas/turkiye.csv",
    "Romanya": "Datas/romanya.csv",
    "Libya": "Datas/libya.csv",
    "KKTC": "Datas/kktc.csv",
    "Azerbaycan": "Datas/azerbaycan.csv"
}

df_global = None

for ulke, dosya_yolu in ulkeler_ve_dosyalar.items():
    df_temp = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "iso-8859-1") \
    .csv(dosya_yolu, header=True, inferSchema=True)
    
    if df_global is None:
        df_global = df_temp
    else:
        df_global = df_global.unionByName(df_temp, allowMissingColumns=True)


toplam_satir = df_global.count()

bitis_zamani = time.time()
gecen_sure = bitis_zamani - baslangic_zamani

print("-" * 50)
print(f"Toplam Satır Sayısı: {toplam_satir}")
print(f"PySpark İşlem Süresi: {gecen_sure:.3f} saniye")
print("-" * 50)

df_global.show(5)

In [ ]:
df_global.printSchema()
tarama_ifadeleri = []
for sutun_adi, sutun_tipi in df_global.dtypes:
    
    if sutun_tipi in ['double', 'float']:
        tarama_ifadeleri.append(F.count(F.when(F.isnan(sutun_adi) | F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))
    else:
        
        tarama_ifadeleri.append(F.count(F.when(F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))
df_global.select(*tarama_ifadeleri).show()


print("TEMEL İSTATİSTİKLER:")
df_global.describe().show()

In [ ]:
df_global = df_global.withColumn("Gercek_Tarih", F.to_date(F.col("yilaygun").cast("string"), "yyyyMMdd"))
df_global = df_global.withColumn("Haftanin_Gunu", F.dayofweek(F.col("Gercek_Tarih")))
df_global = df_global.withColumn("Yilin_Ayi", F.month(F.col("Gercek_Tarih")))   
df_global = df_global.withColumn("Yilin_Haftasi", F.weekofyear("Gercek_Tarih"))
df_global = df_global.withColumn("magazaacilissaati", F.try_to_timestamp(F.try_to_timestamp(F.col("magazaacilissaati"), F.lit("M/d/yyyy H:mm"))))
df_global = df_global.withColumn("magazakapanissaati", F.try_to_timestamp(F.try_to_timestamp(F.col("magazakapanissaati"), F.lit("M/d/yyyy H:mm"))))
df_global = df_global.withColumn("Acik_Kalma_Suresi_Saat",F.round((F.col("magazakapanissaati").cast("long") - F.col("magazaacilissaati").cast("long")) / 3600, 2))
df_global.select("yilaygun", "Gercek_Tarih", "Haftanin_Gunu", "magazaacilissaati", "magazakapanissaati", "Acik_Kalma_Suresi_Saat", "Yilin_Ayi", "Yilin_Haftasi").show(5)

In [ ]:
tarama_ifadeleri = []
for sutun_adi, sutun_tipi in df_global.dtypes:
    if sutun_tipi in ['double', 'float', 'integer', 'long']:
        tarama_ifadeleri.append(F.count(F.when(F.isnan(sutun_adi) | F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))
    else:
        tarama_ifadeleri.append(F.count(F.when(F.col(sutun_adi).isNull(), sutun_adi)).alias(sutun_adi))


df_eksik = df_global.select(*tarama_ifadeleri)


eksik_sozluk = df_eksik.collect()[0].asDict()
sorunlu_sutunlar = {k: v for k, v in eksik_sozluk.items() if v > 0}

print("-" * 50)
if len(sorunlu_sutunlar) == 0:
    print("Veri setinde hiç eksik (Null) değer yok!")
else:
    print("Aşağıdaki sütunlarda eksik veriler tespit edildi:")
    for sutun, adet in sorunlu_sutunlar.items():
        print(f" -> {sutun}: {adet} satır kayıp. Kayıp yüzdesi {adet/(df_global.count()):.2%}")
print("-" * 50)

In [ ]:
from pyspark.ml.feature import Imputer
DROP_COLS = [
    "total_cover", "reyon_cover", 
    "ilkpsf", "sonpsf", 
    "magazam2", "magazamusterigirissayisi",
    "magazaacilissaati", "magazakapanissaati",
    "urunklasmantanim","merchyil","merchyilay","merchyilhafta","yilaygun","magaza","Gercek_Tarih"
]
df_clean = df_global.drop(*DROP_COLS)
df_clean = df_clean.fillna({"indirimorani": 0.0})
imputer = Imputer(
    inputCols=["Acik_Kalma_Suresi_Saat"],
    outputCols=["Acik_Kalma_Suresi_Saat"], 
    strategy="median"
)
df_clean = imputer.fit(df_clean).transform(df_clean)

print("Temizlik Tamamlandı!")

In [ ]:
df_clean = df_clean.withColumn("Haftasonu_Mu", F.when(F.col("Haftanin_Gunu").isin(1, 7), 1).otherwise(0))
df_clean = df_clean.withColumn("Indirim_x_Haftasonu", F.col("indirimorani") * F.col("Haftasonu_Mu"))

In [ ]:
CATEGORİCAL_COLS = [col for col, dtype in df_clean.dtypes if dtype == "string"]
#Cardinality kontrolü
print("\nKategorik Sütunların Kardinalitesi (Benzersiz Değer Sayısı):")
for sutun in CATEGORİCAL_COLS:
    benzersiz_sayisi = df_clean.select(sutun).distinct().count()
    print(f" -> {sutun}: {benzersiz_sayisi} benzersiz değer")

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

print("Kategorik Veri Kodlama (Encoding)")

# 2. KODLANACAK SÜTUNLAR LİSTESİ
kategorik_sutunlar = [
    'ulke', 
    'magazakod', 
    'merchmarkayasgrupkod', 
    'merchaltgrupkod', 
    'depoyerlesimtip'
]

indexers = [
    StringIndexer(
        inputCol=sutun, 
        outputCol=sutun + "_Index", 
        handleInvalid="keep" 
    )
    for sutun in kategorik_sutunlar
]


pipeline_kategorik = Pipeline(stages=indexers)
df_ml_hazir = pipeline_kategorik.fit(df_clean).transform(df_clean)


df_ml_hazir = df_ml_hazir.drop(*kategorik_sutunlar)

print("Tüm kategorik metinler başarıyla matematiksel ID'lere dönüştürüldü!")
df_ml_hazir.printSchema()

In [ ]:
from pyspark.ml.feature import VectorAssembler

print("Makine Öğrenmesi Ön Hazırlığı: Vektör Pres")


bagimsiz_degiskenler = [
    'urunklasmankod', 
    'satisadet_lfl_gy',      
    'indirimorani', 
    'gunlukminimumsicaklik', 
    'gunlukortalamasicaklik', 
    'gunlukmaksimumsicaklik', 
    'ozelgunflag', 
    'gunsonutoplamstok', 
    'gunsonureyonstok', 
    'Haftanin_Gunu', 
    'Yilin_Ayi',              # Sezonsallık
    'Yilin_Haftasi',          # Sezonsallık
    'Acik_Kalma_Suresi_Saat', 
    'ulke_Index', 
    'magazakod_Index', 
    'merchmarkayasgrupkod_Index', 
    'merchaltgrupkod_Index', 
    'depoyerlesimtip_Index',
    'Haftasonu_Mu',
    'Indirim_x_Haftasonu'
]


assembler = VectorAssembler(
    inputCols=bagimsiz_degiskenler,
    outputCol="features", 
    handleInvalid="skip"  
)

# Veriyi dönüştür
df_vektor = assembler.transform(df_ml_hazir)
train_df, test_df = df_vektor.randomSplit([0.8, 0.2], seed=42)

print("-" * 50)
print(f"Eğitim Verisi (Train) Satır Sayısı: {train_df.count()}")
print(f"Test Verisi (Test) Satır Sayısı:   {test_df.count()}")
print("-" * 50)


train_df.select("features", "satisadet").show(5, truncate=False)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

kullanilacak_sutunlar = bagimsiz_degiskenler + ["satisadet"]

pdf_train = train_df.select(kullanilacak_sutunlar).toPandas()
pdf_test = test_df.select(kullanilacak_sutunlar).toPandas()

# X (Özellikler) ve Y (Hedef) olarak ayırıyoruz
X_train = pdf_train.drop("satisadet", axis=1)
y_train = pdf_train["satisadet"]

X_test = pdf_test.drop("satisadet", axis=1)
y_test = pdf_test["satisadet"]

print("Veri yerel belleğe alındı.")



lgbm_model = lgb.LGBMRegressor(
    n_estimators=150,       # 100 Ağaçlık bir orman
    learning_rate=0.1,      # Öğrenme hızı
    max_depth=16,           # Maximum derinlik
    random_state=42,
    n_jobs=-1               # TÜM işlemci çekirdeklerini kullan
)


lgbm_model.fit(X_train, y_train)
y_pred = lgbm_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE (Ortalama Sapma) : {rmse:.2f} adet")
print(f"R-Squared (Başarı)    : {r2:.4f} ")
print("Model En Çok Hangi Bilgilerden Öğrendi? (Feature Importance)")
onem_dereceleri = list(zip(X_train.columns, lgbm_model.feature_importances_))
# En önemliden en önemsize doğru sırala
onem_dereceleri.sort(key=lambda x: x[1], reverse=True)

for sutun, skor in onem_dereceleri[:5]:
    print(f" -> {sutun}: {skor} puan")

In [ ]:
import joblib
joblib.dump(lgbm_model, "lightgbm_model.pkl")
print("Model diske kaydedildi: 'lightgbm_model.pkl'")

In [ ]:
spark.stop()
print("Spark oturumu kapatıldı. Tüm işlemler tamamlandı!")